# Entrenamiento de los modelos YOLO

Entrena los dos detectores del proyecto con `src/train/train.py` y la configuración común `configs/train_base.yaml`:

| Modelo | Datos | Entregables |
|---|---|---|
| `full_yolov8s` | CCT completo (`data_full.yaml`) | E1, E2, E4, E5 |
| `eccv_yolov8s` | Partición oficial ECCV (`data_eccv.yaml`) | E3 |

Las corridas se guardan en Drive (`MyDrive/redes-parcial-p3/runs/`) para no perderlas si se cierra la sesión y poder reanudarlas.

Usar un entorno con GPU **L4** o **A100** (más disco y más rápido que la T4).

## Preparar la máquina

Cada notebook de Colab corre en su propia máquina, así que primero se ejecuta `colab_setup.ipynb` completo: monta Drive, clona el repo, instala dependencias, extrae las imágenes y genera las etiquetas YOLO. La extracción de los 105 GB tarda bastante.

In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')
REPO_DIR = '/content/redes-parcial-p3'
if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/Roxana-Vargas/redes-parcial-p3.git {REPO_DIR}
%cd {REPO_DIR}
!git pull
%run notebooks/colab_setup.ipynb

In [ ]:
PROJECT = '/content/drive/MyDrive/redes-parcial-p3/runs'
!cat configs/train_base.yaml

## Corrida de prueba

Una época con el 2 % del train, para verificar que los datos, las etiquetas y la GPU funcionan antes de lanzar el entrenamiento largo. Si falla aquí, no se pierden horas de GPU.

In [ ]:
!python src/train/train.py --data data/yolo/data_full.yaml --name prueba --project {PROJECT} --epochs 1 --fraction 0.02

## Modelo principal (CCT completo)

Entrenamiento largo: unas 113k imágenes por época (83 % vacías). Revisar de vez en cuando que la sesión siga activa.

In [ ]:
!python src/train/train.py --data data/yolo/data_full.yaml --name full_yolov8s --project {PROJECT}

## Modelo E3 (partición oficial ECCV)

Misma configuración, entrenado solo con `eccv_train` (unas 11k imágenes). Se valida en `eccv_cis_val`.

In [ ]:
!python src/train/train.py --data data/yolo/data_eccv.yaml --name eccv_yolov8s --project {PROJECT}

## Reanudar una corrida cortada

Si la sesión de Colab se desconecta a mitad del entrenamiento, volver a preparar la máquina (primera celda) y reanudar desde el último checkpoint guardado en Drive. Cambiar `RUN` por la corrida que se cortó.

In [ ]:
RUN = 'full_yolov8s'
!python src/train/train.py --name {RUN} --project {PROJECT} --resume

## Curvas de entrenamiento

Pérdidas y métricas por época que guarda Ultralytics en cada corrida.

In [ ]:
from IPython.display import Image, display

for run in ['full_yolov8s', 'eccv_yolov8s']:
    path = os.path.join(PROJECT, run, 'results.png')
    if os.path.exists(path):
        print(run)
        display(Image(filename=path, width=900))